#### Ingesting FHV Trips Data

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from dotenv import load_dotenv
import logging
import os

load_dotenv()

##### Creating directory for fhv log file 

In [ ]:
log_file = r"/app/data/logs/"
os.makedirs(name = log_file,exist_ok= True)
file_name_fhv = os.path.join(log_file,'fhv_trips.log')
print(file_name_fhv)

##### Adding logger

In [ ]:
logger = logging.getLogger(__name__)
logger.propagate = False   # <-- add this
logger.setLevel(logging.DEBUG)
fh = logging.FileHandler(file_name_fhv,mode = 'w')
logger.addHandler(fh)
formatter = logging.Formatter('[%(asctime)s] %(levelname)s: %(message)s')
fh.setFormatter(formatter)

##### Creating spark session

In [ ]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("NYC Taxi Pipeline")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.5.0,"
        "software.amazon.awssdk:bundle:2.31.54"
    )
    .config("spark.driver.memory", "4g")
    .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.endpoint", os.environ.get("S3_ENDPOINT", "s3.amazonaws.com"))
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions",8)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version : {spark.version}")
print(f"App name      : {spark.sparkContext.appName}")
print(f"Master        : {spark.sparkContext.master}")

##### Create Schema Definition
 

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType
)

fhv_schema = StructType([
    StructField("dispatching_base_num", StringType(), True),
    StructField("pickup_datetime", TimestampType(), True),
    StructField("dropOff_datetime", TimestampType(), True),
    StructField("PUlocationID", LongType(), True),
    StructField("DOlocationID", LongType(), True),
    StructField("SR_Flag", LongType(), True),
    StructField("Affiliated_base_number", StringType(), True)
])

##### Reading FHV file

In [ ]:
fhv_data = spark.read\
                .option('header',True)\
                    .schema(fhv_schema)\
                        .parquet('/app/data/input/fhv/fhv_tripdata_2026-04.parquet')

In [ ]:
fhv_data.printSchema()

In [ ]:
fhv_data.show(10)

##### Log Raw Count

In [ ]:
logger.info(f'Raw count: {fhv_data.count()}')

##### Cleansing data outside of April 2026 Window

In [ ]:
fhv_data = fhv_data.filter(~(date_format(col('pickup_datetime'),'yyyy-MM') >= '2026-05'))

In [ ]:
fhv_data = fhv_data.filter(~(date_format(col('pickup_datetime'),'yyyy-MM') < '2026-04'))

##### Removing data if location data is missing

In [ ]:
fhv_data = fhv_data.filter(~(col('PUlocationID').isNull() & col('DOlocationID').isNull()))

##### Checking if pickup and drop datetime are identical (data quality issue)

In [ ]:
fhv_DQ_cnt = fhv_data.filter(col('pickup_datetime') >= col('dropOff_datetime')).count()

In [ ]:
if fhv_DQ_cnt != 0:
    fhv_data = fhv_data.filter(~(col('pickup_datetime') == col('dropOff_datetime')))


##### Log count after validation

In [ ]:
logger.info(f"After Validation Count: {fhv_data.count()}")

##### Derived Column trip duration in minutes

In [ ]:
from pyspark.sql.functions import timestamp_diff
fhv_data = fhv_data.withColumn('trip_duration_minutes',timestamp_diff('minute',col('pickup_datetime'),col('dropOff_datetime')))

##### Clearing any trips with zero minutes trip duration

In [ ]:
fhv_data = fhv_data.filter(~(col('trip_duration_minutes') == 0))

##### Adding source filename and ingestion timestamp columns

In [ ]:
fhv_data = fhv_data.withColumns({'source_file': lit('fhv_tripdata_2026-04.parquet'),
'ingestion_timestamp' : current_timestamp()})

##### Writing data with paritioning in S3 Bucket

In [ ]:
s3_bucket = os.environ["S3_BUCKET"]

fhv_data \
    .withColumn('pickup_date', date_format(col('pickup_datetime'), 'yyyy-MM-dd')) \
    .write \
    .option('header', True) \
    .partitionBy('pickup_date') \
    .mode('overwrite') \
    .parquet(f's3a://{s3_bucket}/fhv')

##### Data Count written to S3

In [ ]:
logger.info(f'Complete data count written to S3: {fhv_data.count()}')